# PosteriorEstimation using NumPyro

In [3]:
import astropy.units as u
import jax
import jax.numpy as jnp
import numpy as np
import numpyro
import numpyro.distributions as dist
from astropy.coordinates import EarthLocation, SkyCoord
from astropy.time import Time
from numpyro.infer import MCMC, NUTS

from nyx.atmosphere import single_scattering
from nyx.core import Scene
from nyx.core.geometry import Geometry
from nyx.core.observation import Observation
from nyx.emitter import Airglow, Moon, Stars, ZodiacalLight
from nyx.instrument import EffectiveApertureInstrument

JAX is not using 64-bit precision. Results will diverge from healpy even at moderate nside and can overflow for nside > 8192. See the README on numerical precision.


## Building Model:

In [4]:
geo = Geometry(
    wvls=jnp.linspace(300, 700, 40) * u.nm,  # wavelength grid
    nside=32,  # HEALPix resolution
    ngrid=2,  # FOV eval grid
    fov=3.5 * u.deg,  # field of view
)

hess_ct1 = EffectiveApertureInstrument.load("../../nyx/data/HESS_CT1.h5", geo)
hess_stereo = {"CT1": hess_ct1, "CT2": hess_ct1, "CT3": hess_ct1, "CT4": hess_ct1}
atmosphere = single_scattering.HGOzoneAbsorption(geo)

sources = {
    "airglow": Airglow.from_eso_skycalc(geo, sfu=100.0),
    "zodiacal": ZodiacalLight.from_leinert1998(geo),
    "GaiaDR3": Stars.from_gaia_dr3(geo, lim_mag=12),
    "moon": Moon.from_jones2013(geo),
}

## Creating scenes:

In [5]:
location_ct1 = EarthLocation.from_geodetic(
    lon=16.50091 * u.deg,
    lat=-23.27153 * u.deg,
    height=1.835 * u.km,
)
location_ct2 = EarthLocation.from_geodetic(
    lon=16.50010 * u.deg,
    lat=-23.27075 * u.deg,
    height=1.835 * u.km,
)
location_ct3 = EarthLocation.from_geodetic(
    lon=16.49925 * u.deg,
    lat=-23.27151 * u.deg,
    height=1.835 * u.km,
)
location_ct4 = EarthLocation.from_geodetic(
    lon=16.50012 * u.deg,
    lat=-23.27232 * u.deg,
    height=1.835 * u.km,
)

times = Time("2021-01-12T21:12:16", scale="utc") + jnp.linspace(0, 30, 3) * 60 * u.s
target = SkyCoord(l=184 * u.deg, b=-6 * u.deg, frame="galactic")

obs_ct1 = Observation(location_ct1, times, target, geo)
obs_ct2 = Observation(location_ct2, times, target, geo)
obs_ct3 = Observation(location_ct3, times, target, geo)
obs_ct4 = Observation(location_ct4, times, target, geo)

observations = {"CT1": obs_ct1, "CT2": obs_ct2, "CT3": obs_ct3, "CT4": obs_ct4}

## Generating synthetic truth data:

In [6]:
scene = Scene.build(hess_stereo, atmosphere, sources, observations)

In [7]:
# Apply known per-observation shifts as ground truth
scene_shifted = scene
true_shifts = {}
for i, name in enumerate(["CT1", "CT2", "CT3", "CT4"]):
    rng = np.random.default_rng(42 + i)
    shifts = np.deg2rad(rng.uniform(-0.05, 0.05, size=(scene.nobs[name], 2)))
    true_shifts[name] = shifts
    scene_shifted = scene_shifted.set(f"{name}.shift", jnp.array(shifts))

# Render truth images
truth_rates = scene_shifted.render()
rel_error = 0.1
truth_images = {
    name: tr * rng.lognormal(mean=0, sigma=rel_error, size=tr.shape)
    for name, tr in truth_rates.items()
}

# Diagnostics:

## Numpyro model

In [8]:
def nyx_model(truth_images):
    params = {}
    params["atmosphere.Mie.aod_500"] = numpyro.sample("aod_500", dist.Uniform(0.001, 2.0))
    params["atmosphere.Mie.angstrom_exp"] = numpyro.sample("angstrom_exp", dist.Uniform(0.0, 2.5))
    params["atmosphere.Mie.hg_asymmetry"] = numpyro.sample(
        "hg_asymmetry", dist.TruncatedNormal(loc=0.65, scale=0.1, low=0.5, high=0.9)
    )
    rel_error = numpyro.sample("rel_error", dist.HalfNormal(0.1))

    for name in scene.instruments:
        with numpyro.plate(f"obs_{name}", scene.nobs[name]):
            xs = numpyro.sample(f"{name}.xshift", dist.Uniform(jnp.deg2rad(-0.1), jnp.deg2rad(0.1)))
            ys = numpyro.sample(f"{name}.yshift", dist.Uniform(jnp.deg2rad(-0.1), jnp.deg2rad(0.1)))
            rot = numpyro.sample(
                f"{name}.rotation", dist.Uniform(jnp.deg2rad(-1.0), jnp.deg2rad(1.0))
            )
        params[f"{name}.shift"] = jnp.stack([xs, ys], axis=-1)
        params[f"{name}.rotation"] = rot[:, None]
        params[f"{name}.efficiency"] = numpyro.sample(f"{name}.efficiency", dist.Uniform(0.5, 2.0))

    predicted = scene.set_params(params).render()

    for name, obs in truth_images.items():
        with numpyro.plate(f"obs_ll_{name}", obs.shape[0]):
            numpyro.sample(
                f"images_{name}",
                dist.LogNormal(jnp.log(predicted[name]), rel_error).to_event(1),
                obs=obs,
            )

## Running NUTS

In [9]:
print("Running NUTS ...")

kernel = NUTS(
    nyx_model,
    max_tree_depth=6,
    dense_mass=[
        (
            "aod_500",
            "angstrom_exp",
            "hg_asymmetry",
            "CT1.efficiency",
            "CT2.efficiency",
            "CT3.efficiency",
            "CT4.efficiency",
            "rel_error",
        )
    ],
)
mcmc = MCMC(kernel, num_warmup=200, num_samples=500, num_chains=1)
mcmc.run(jax.random.PRNGKey(0), truth_images=truth_images, extra_fields=("num_steps",))
mcmc.print_summary()

num_steps = mcmc.get_extra_fields()["num_steps"]
print(f"Leapfrog steps per sample — mean: {num_steps.mean():.1f}, max: {num_steps.max()}")

samples = mcmc.get_samples()

print("\n Posterior summary ")
for name in [
    "aod_500",
    "angstrom_exp",
    "hg_asymmetry",
    "CT1.efficiency",
    "CT2.efficiency",
    "CT3.efficiency",
    "CT4.efficiency",
]:
    s = samples[name]
    print(f"  {name:20s}:  mean={s.mean():.4f}  std={s.std():.4f}")

for tel in ["CT1", "CT2", "CT3", "CT4"]:
    shift_samples = jnp.stack([samples[f"{tel}.xshift"], samples[f"{tel}.yshift"]], axis=-1)
    shift_mean_deg = np.rad2deg(shift_samples.mean(axis=0))
    comp_shifts = true_shifts[tel]
    true_deg = np.rad2deg(comp_shifts)
    residual_arcsec = (shift_mean_deg - true_deg) * 3600
    print(f"\n  {tel} shift residual RMS: {np.std(residual_arcsec):.1f} arcsec")

Running NUTS ...


sample: 100%|█| 700/700 [08:15<00:00,  1.41it/s, 15 steps of size 3.29e-01. acc. p



                      mean       std    median      5.0%     95.0%     n_eff     r_hat
  CT1.efficiency      1.00      0.01      1.00      0.99      1.01    472.20      1.00
 CT1.rotation[0]      0.00      0.00      0.00     -0.00      0.00    955.78      1.00
 CT1.rotation[1]      0.00      0.00      0.00      0.00      0.00    650.75      1.00
 CT1.rotation[2]     -0.00      0.00     -0.00     -0.00      0.00    645.60      1.00
   CT1.xshift[0]      0.00      0.00      0.00      0.00      0.00    959.03      1.00
   CT1.xshift[1]      0.00      0.00      0.00      0.00      0.00    683.73      1.00
   CT1.xshift[2]     -0.00      0.00     -0.00     -0.00     -0.00   1004.35      1.00
   CT1.yshift[0]     -0.00      0.00     -0.00     -0.00     -0.00   1300.44      1.00
   CT1.yshift[1]      0.00      0.00      0.00      0.00      0.00    834.89      1.00
   CT1.yshift[2]      0.00      0.00      0.00      0.00      0.00    983.33      1.00
  CT2.efficiency      1.01      0.01      